# Samostatná práce: experimenty s klasifikací, regresí a shlukováním

Samostatná práce na druhou polovinu cvičení – nehodnotí se. Pracujte sami nebo ve dvojicích, a když se zaseknete, zeptejte se.

Všechny buňky jdou spustit hned. Úkoly říkají, co v kódu **upravit** (řádky označené `# UPRAVIT`); po úpravě buňku spusťte znovu a podívejte se, jak se změnil výsledek.
Otázky na konci probereme společně.

Data **nepředzpracováváme** (škálování apod.) – to je téma cvičení 08.

## 1. Příprava dat – Wine

Chemická analýza 178 vín ze tří odrůd (třídy 0, 1, 2), 13 číselných příznaků.

Data rozdělíme na **trénovací** (75 %) a **validační** (25 %) část. Dnes nic nevybíráme „naostro“, takže testovací množinu nepotřebujeme –
postup s výběrem modelu a konečným testem přijde na cvičení 07.

In [ ]:
import numpy as np                     # pole a matematické operace
import pandas as pd                    # tabulky
import matplotlib.pyplot as plt        # grafy
from IPython.display import display    # zobrazení tabulek

from sklearn import datasets, metrics
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

RSTATE = 42                                                       # opakovatelné výsledky

wine = datasets.load_wine(as_frame=True)
X, y = wine.data, wine.target                                     # příznaky (DataFrame) a odrůda (Series)
X_train, X_val, y_train, y_val = train_test_split(                # jedno společné dělení X a y
    X, y, test_size=0.25, stratify=y, random_state=RSTATE
)
df = X_train.assign(target=y_train)                               # tabulka pro prohlížení: trénovací data + cíl
print("Trénovací:", X_train.shape, " Validační:", X_val.shape)

## 2. Prohlížení dat v pandas

**Úkol A – výběr řádků a sloupců**
1. Změňte `LIMIT_ALKOHOLU` ze 14 na 13.
2. Přidejte do `SLOUPCE` příznak `"flavanoids"`.
3. Rozšiřte masku o podmínku `df["proline"] > 800` – nejprve pomocí `&`, potom pomocí `|`. Jak se změní počet řádků?
4. Seřaďte zobrazený výběr sestupně podle `proline`: `.sort_values("proline", ascending=False)`.

In [ ]:
LIMIT_ALKOHOLU = 14                                     # UPRAVIT: hranice obsahu alkoholu
SLOUPCE = ["alcohol", "proline", "target"]              # UPRAVIT: zobrazené sloupce
maska = df["alcohol"] > LIMIT_ALKOHOLU                  # UPRAVIT: přidat podmínku na proline
vyber = df.loc[maska, SLOUPCE]                          # řádky podle masky, vybrané sloupce
print("Vybraných řádků:", maska.sum())
display(vyber.head(8))                                  # UPRAVIT: před .head() přidat řazení

**Úkol B – souhrny**
1. Doplňte zobrazení **podílů** tříd pomocí `value_counts(normalize=True)`.
2. Seřaďte tabulku rozsahů sestupně podle sloupce `rozsah`. Který příznak má zdaleka největší rozsah?

In [ ]:
display(y_train.value_counts().sort_index().rename("počet"))    # kolik vín má každá odrůda
# UPRAVIT: sem přidat podíly tříd

rozsahy = X_train.agg(["min", "max"]).T                          # minimum a maximum každého příznaku
rozsahy["rozsah"] = rozsahy["max"] - rozsahy["min"]              # nový sloupec: rozdíl max - min
display(rozsahy.round(2))                                        # UPRAVIT: seřadit podle rozsahu

## 3. Klasifikace – příznaky a složitost modelu

**Úkol C – experiment s rozhodovacím stromem**
1. Přidejte do `SADY_PRIZNAKU` sadu `"jen_alcohol": ["alcohol"]`.
2. Rozšiřte `HLOUBKY` na všechny hodnoty 1–10 (`list(range(1, 11))`).
3. Spusťte obě následující buňky. Kterou sadu příznaků a hloubku byste si tipli jako nejlepší? Je model někde přeučený?

Všechny varianty používají stejné trénovací a validační rozdělení. Cílový sloupec `target` do sady příznaků nepatří.

In [ ]:
SADY_PRIZNAKU = {                                        # UPRAVIT: název sady -> seznam příznaků
    "všechny": list(X_train.columns),                    # všech 13 měření
    "dva": ["alcohol", "flavanoids"],                    # menší sada pro porovnání
}
HLOUBKY = [1, 2, 3, 5]                                   # UPRAVIT: zkoušené maximální hloubky

baseline_acc = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).score(X_val, y_val)
rows = []
for sada, sloupce in SADY_PRIZNAKU.items():              # každá sada příznaků ...
    for hloubka in HLOUBKY:                              # ... s každou hloubkou
        tree = DecisionTreeClassifier(max_depth=hloubka, random_state=RSTATE)
        tree.fit(X_train[sloupce], y_train)              # učení jen na vybraných sloupcích
        rows.append({"sada": sada, "hloubka": hloubka,
                     "train_accuracy": tree.score(X_train[sloupce], y_train),
                     "val_accuracy": tree.score(X_val[sloupce], y_val)})
vysledky = pd.DataFrame(rows)
display(vysledky.round(3))
print("Validační správnost baseline:", round(baseline_acc, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for sada, skupina in vysledky.groupby("sada"):           # jedna křivka pro každou sadu příznaků
    ax.plot(skupina["hloubka"], skupina["val_accuracy"], "o-", label=sada)
ax.axhline(baseline_acc, color="gray", linestyle="--", label="baseline")   # referenční úroveň
ax.set(xlabel="max_depth", ylabel="validační správnost", ylim=(0, 1.05))
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. Regrese – Diabetes

**Úkol D – číselné předpovědi**
1. Rozšiřte `REG_HLOUBKY` na `[1, 2, 3, 5, 10]`.
2. Do `REG_SADY` přidejte sadu `"bmi": ["bmi"]`.
3. Seřaďte výslednou tabulku vzestupně podle `val_MAE`. Který model má nejmenší chybu? A který je jen o málo lepší než průměr?

In [ ]:
diabetes = datasets.load_diabetes(as_frame=True)
Xr_train, Xr_val, yr_train, yr_val = train_test_split(           # trénovací a validační část
    diabetes.data, diabetes.target, test_size=0.25, random_state=RSTATE
)
REG_HLOUBKY = [1, 3]                                             # UPRAVIT: hloubky regresních stromů
REG_SADY = {"všechny": list(Xr_train.columns)}                   # UPRAVIT: přidat sadu jen s bmi

rows = []
for sada, sloupce in REG_SADY.items():
    modely = [("baseline (průměr)", DummyRegressor(strategy="mean")), ("lineární", LinearRegression())]
    modely += [(f"strom {d}", DecisionTreeRegressor(max_depth=d, random_state=RSTATE)) for d in REG_HLOUBKY]
    for nazev, model in modely:
        model.fit(Xr_train[sloupce], yr_train)                   # učení na trénovacích datech
        pred = model.predict(Xr_val[sloupce])                    # předpovědi validačních dat
        rows.append({"sada": sada, "model": nazev,
                     "train_MAE": metrics.mean_absolute_error(yr_train, model.predict(Xr_train[sloupce])),
                     "val_MAE": metrics.mean_absolute_error(yr_val, pred),
                     "val_RMSE": metrics.root_mean_squared_error(yr_val, pred)})
reg_vysledky = pd.DataFrame(rows)
display(reg_vysledky.round(1))                                   # UPRAVIT: seřadit podle val_MAE

## 5. Shlukování – Iris

**Úkol E – skupiny bez známých cílů**
1. Rozšiřte `K_VALUES` na všechna celá čísla 2–8.
2. Do `CLUSTER_SADY` přidejte sadu `"okvětní": ["petal length (cm)", "petal width (cm)"]`.
3. Přidejte do tabulky sloupec s velikostí nejmenšího shluku: `"nejmenší": pd.Series(labels).value_counts().min()`.

In [ ]:
iris = datasets.load_iris(as_frame=True)
X_iris = iris.data                                               # jen měření, druhy K-means nedostane
K_VALUES = [2, 3, 5]                                             # UPRAVIT: počty shluků 2–8
CLUSTER_SADY = {"všechny": list(X_iris.columns)}                 # UPRAVIT: přidat sadu okvětních lístků

rows = []
for sada, sloupce in CLUSTER_SADY.items():
    for k in K_VALUES:
        km = KMeans(n_clusters=k, n_init=10, random_state=RSTATE)
        labels = km.fit_predict(X_iris[sloupce])                 # učení bez cílové proměnné
        rows.append({"sada": sada, "k": k,                       # UPRAVIT: přidat velikost nejmenšího shluku
                     "silhouette": metrics.silhouette_score(X_iris[sloupce], labels)})
cluster_vysledky = pd.DataFrame(rows)
display(cluster_vysledky.round(3))

ax = cluster_vysledky.pivot(index="k", columns="sada", values="silhouette").plot(marker="o", figsize=(8, 4))
ax.set(xlabel="počet shluků k", ylabel="silhouette"); ax.grid(alpha=0.3)
plt.show()

**Úkol F – shlukování vín**

Spusťte K-means se 3 shluky na víně (`X`) a porovnejte shluky se skutečnými odrůdami pomocí Adjusted Rand Index. Stromy měly vysokou validační správnost – proč shlukování odpovídá odrůdám mnohem hůř?
Podívejte se znovu na tabulku rozsahů z úkolu B. (Opravíme to na cvičení 08.)

In [ ]:
km_wine = KMeans(n_clusters=3, n_init=10, random_state=RSTATE)
labels_wine = km_wine.fit_predict(X)                             # shluky vín bez odrůd
print("ARI vína:", round(metrics.adjusted_rand_score(y, labels_wine), 3))
# UPRAVIT (dobrovolně): zkuste shlukovat jen podle sloupce "proline" – jak se změní ARI?

## Otázky ke společnému rozboru

1. Jak se změnil výběr řádků při použití `&` a `|`?
2. Který příznak vína má největší rozsah? Znamená to, že je nejužitečnější?
3. Jak se změnila validační správnost při změně sady příznaků? Stačí jediný příznak `alcohol`?
4. Pro které hloubky je strom podučený? Proč se nerozhodujeme podle trénovací správnosti?
5. Čím se liší regresní metriky (MAE, RMSE) od správnosti u klasifikace? Proč může RMSE vyjít víc než MAE?
6. Jak dopadl model jen s `bmi` ve srovnání s baseline a se všemi příznaky?
7. Souhlasí nejvyšší silhouette se skutečným počtem druhů kosatců (3)?
8. Proč shlukování vín tolik nesouhlasí s odrůdami?

[Zpět k hlavnímu notebooku](ML_Introduction_CZ.ipynb)